In [183]:
import pandas as pd

In [184]:
def normalize_to_month_start(df, date_col):
    # Chuyển về datetime nếu chưa phải
    df[date_col] = pd.to_datetime(df[date_col])
    # Quy chuẩn: Tháng 12/2024 (bất kể ngày 01 hay 31) -> 2024-12-01
    df[date_col] = df[date_col].dt.to_period('M').dt.to_timestamp()
    return df

In [185]:
emberVN = pd.read_csv(r"data\raw\ember_data\emberVN.csv")
emberVN = normalize_to_month_start(emberVN, 'date')

In [186]:
emberVN.drop(columns=["generation_share_yoy_change_pct_points","generation_yoy_change_pct",
                      "generation_yoy_change_twh","generation_yoy_change_kwh_per_capita"], inplace=True)

In [187]:
emberVN.head()

,entity,entity_code,is_aggregate_entity,date,series,is_aggregate_series,generation_twh,generation_share_pct,generation_kwh_per_capita,generation_ytd_twh,generation_ytd_share_pct,generation_ytd_kwh_per_capita
0,Viet Nam,VNM,False,2019-01-01,Coal,False,9.63,53.89,99.10,9.63,53.89,99.10
1,Viet Nam,VNM,False,2019-01-01,Gas,False,4.12,23.06,42.40,4.12,23.06,42.40
2,Viet Nam,VNM,False,2019-01-01,Hydro,False,3.39,18.97,34.89,3.39,18.97,34.89
3,Viet Nam,VNM,False,2019-01-01,Other fossil,False,0.00,0.00,0.00,0.00,0.00,0.00
4,Viet Nam,VNM,False,2019-01-01,Solar,False,0.00,0.00,0.00,0.00,0.00,0.00


In [188]:
IPI_VN = pd.read_excel(r"data\raw\gso_data\IPI_(2019-2025).xlsx")
IPI_VN = IPI_VN.rename(columns={'Unnamed: 0': 'Year'})

# Chuẩn hoá Year về số nguyên (không parse như datetime ở bước này)
IPI_VN['Year'] = pd.to_numeric(IPI_VN['Year'], errors='coerce').astype('Int64')
IPI_VN = IPI_VN[IPI_VN['Year'].notna()].copy()
IPI_VN['Year'] = IPI_VN['Year'].astype(int)

# 3. Sử dụng hàm melt để chuyển các cột "Tháng X" thành dòng
IPI_long = IPI_VN.melt(
    id_vars=['Year'],
    var_name='Month_Raw',
    value_name='IPI_Value'
)

# 4. Giữ lại các cột tháng hợp lệ và tách Month
IPI_long = IPI_long[IPI_long['Month_Raw'].astype(str).str.match(r'^Tháng\s+\d+$')].copy()
IPI_long['Month'] = IPI_long['Month_Raw'].str.replace('Tháng ', '', regex=False).astype(int)
IPI_long['IPI_Value'] = pd.to_numeric(IPI_long['IPI_Value'], errors='coerce')

# 5. Tạo cột date bằng Year + Month + day=1
IPI_long['date'] = pd.to_datetime(dict(year=IPI_long['Year'], month=IPI_long['Month'], day=1))

# 6. Sắp xếp lại dữ liệu theo thời gian và dọn dẹp
IPI_final = IPI_long.sort_values('date').reset_index(drop=True)
IPI_final = IPI_final[['date', 'IPI_Value']]
IPI_final.head(100)

,date,IPI_Value
0,2019-01-01,134.05
1,2019-02-01,108.37
2,2019-03-01,134.77
3,2019-04-01,134.56
4,2019-05-01,138.69
...,...,...
79,2025-08-01,189.16
80,2025-09-01,188.35
81,2025-10-01,191.77
82,2025-11-01,192.43


In [189]:
# combine 2 data
emberVN['date'] = pd.to_datetime(emberVN['date'], format='%Y-%m')
emberVNcombine = pd.merge(emberVN, IPI_final, on='date', how='inner')
emberVNcombine.head()

,entity,entity_code,is_aggregate_entity,date,series,is_aggregate_series,generation_twh,generation_share_pct,generation_kwh_per_capita,generation_ytd_twh,generation_ytd_share_pct,generation_ytd_kwh_per_capita,IPI_Value
0,Viet Nam,VNM,False,2019-01-01,Coal,False,9.63,53.89,99.10,9.63,53.89,99.10,134.05
1,Viet Nam,VNM,False,2019-01-01,Gas,False,4.12,23.06,42.40,4.12,23.06,42.40,134.05
2,Viet Nam,VNM,False,2019-01-01,Hydro,False,3.39,18.97,34.89,3.39,18.97,34.89,134.05
3,Viet Nam,VNM,False,2019-01-01,Other fossil,False,0.00,0.00,0.00,0.00,0.00,0.00,134.05
4,Viet Nam,VNM,False,2019-01-01,Solar,False,0.00,0.00,0.00,0.00,0.00,0.00,134.05


In [190]:
CPI_VN = pd.read_excel(r"data\raw\gso_data\CPI.xlsx")
CPI_VN = CPI_VN.rename(columns={'Unnamed: 0': 'Month_Raw'})

CPI_long = CPI_VN.melt(
    id_vars=['Month_Raw'],
    var_name='Year',
    value_name='CPI_Value'
)

# 4. Giữ lại các dòng tháng hợp lệ "Tháng 1" ... "Tháng 12"
CPI_long = CPI_long[CPI_long['Month_Raw'].astype(str).str.match(r'^Tháng\s+\d+$')].copy()

# 5. Tách tháng và chuẩn hoá kiểu dữ liệu
CPI_long['Month'] = CPI_long['Month_Raw'].str.replace('Tháng ', '', regex=False).astype(int)
CPI_long['Year'] = CPI_long['Year'].astype(int)
CPI_long['CPI_Value'] = pd.to_numeric(
    CPI_long['CPI_Value'].astype(str).str.replace(',', '.', regex=False),
    errors='coerce'
)

# 6. Tạo cột date
CPI_long['date'] = pd.to_datetime(dict(year=CPI_long['Year'], month=CPI_long['Month'], day=1))

# 7. Sắp xếp lại dữ liệu theo thời gian và dọn dẹp
CPI_final = CPI_long.sort_values('date').reset_index(drop=True)
CPI_final = CPI_final[['date', 'CPI_Value']]
CPI_final.head(100)


,date,CPI_Value
0,2019-01-01,100.10
1,2019-02-01,100.80
2,2019-03-01,99.79
3,2019-04-01,100.31
4,2019-05-01,100.49
...,...,...
67,2024-08-01,100.00
68,2024-09-01,100.29
69,2024-10-01,100.33
70,2024-11-01,100.13


In [191]:
# Combine 3 data
emberVNcombine = pd.merge(emberVNcombine, CPI_final, on='date', how='inner')
emberVNcombine.head()

,entity,entity_code,is_aggregate_entity,date,series,is_aggregate_series,generation_twh,generation_share_pct,generation_kwh_per_capita,generation_ytd_twh,generation_ytd_share_pct,generation_ytd_kwh_per_capita,IPI_Value,CPI_Value
0,Viet Nam,VNM,False,2019-01-01,Coal,False,9.63,53.89,99.10,9.63,53.89,99.10,134.05,100.1
1,Viet Nam,VNM,False,2019-01-01,Gas,False,4.12,23.06,42.40,4.12,23.06,42.40,134.05,100.1
2,Viet Nam,VNM,False,2019-01-01,Hydro,False,3.39,18.97,34.89,3.39,18.97,34.89,134.05,100.1
3,Viet Nam,VNM,False,2019-01-01,Other fossil,False,0.00,0.00,0.00,0.00,0.00,0.00,134.05,100.1
4,Viet Nam,VNM,False,2019-01-01,Solar,False,0.00,0.00,0.00,0.00,0.00,0.00,134.05,100.1


In [192]:
GDP_VN = pd.read_excel(r"data\raw\gso_data\GDP.xlsx")
GDP_VN.columns = ['Year', 'GDP_VND_Trillion', 'GDP_USD_Billion', 'GDP_per_Capita_USD']
GDP_VN = GDP_VN.drop(0).reset_index(drop=True) # Bỏ dòng tiêu đề thừa

# Ép kiểu Year về số nguyên để khớp với file 1
GDP_VN['Year'] = GDP_VN['Year'].astype(int)

# Xử lý các ký tự đặc biệt như "~" và chuẩn hoá định dạng số (vd: 8.044,4 -> 8044.4)
GDP_VN['GDP_VND_Trillion'] = pd.to_numeric(
	GDP_VN['GDP_VND_Trillion']
	.astype(str)
	.str.replace('~', '', regex=False)
	.str.replace('.', '', regex=False)
	.str.replace(',', '.', regex=False),
	errors='coerce'
)
GDP_VN.head()

,Year,GDP_VND_Trillion,GDP_USD_Billion,GDP_per_Capita_USD
0,2019,7700.0,"334,4",3.439
1,2020,8044.4,"346,6",3.549
2,2021,8479.7,"366,5",3.757
3,2022,9513.3,"410,3",4.133
4,2023,10221.8,"430,0",4.317


In [193]:
emberVNcombine = pd.merge(emberVNcombine, GDP_VN[['Year', 'GDP_VND_Trillion']],
                           left_on=emberVNcombine['date'].dt.year, right_on='Year', how='inner')
emberVNcombine

,entity,entity_code,is_aggregate_entity,date,series,is_aggregate_series,generation_twh,generation_share_pct,generation_kwh_per_capita,generation_ytd_twh,generation_ytd_share_pct,generation_ytd_kwh_per_capita,IPI_Value,CPI_Value,Year,GDP_VND_Trillion
0,Viet Nam,VNM,False,2019-01-01,Coal,False,9.63,53.89,99.10,9.63,53.89,99.10,134.05,100.10,2019,7700.0
1,Viet Nam,VNM,False,2019-01-01,Gas,False,4.12,23.06,42.40,4.12,23.06,42.40,134.05,100.10,2019,7700.0
2,Viet Nam,VNM,False,2019-01-01,Hydro,False,3.39,18.97,34.89,3.39,18.97,34.89,134.05,100.10,2019,7700.0
3,Viet Nam,VNM,False,2019-01-01,Other fossil,False,0.00,0.00,0.00,0.00,0.00,0.00,134.05,100.10,2019,7700.0
4,Viet Nam,VNM,False,2019-01-01,Solar,False,0.00,0.00,0.00,0.00,0.00,0.00,134.05,100.10,2019,7700.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499,Viet Nam,VNM,False,2024-12-01,Hydro,False,11.29,41.95,111.80,98.13,31.83,971.70,175.15,100.29,2024,11511.9
500,Viet Nam,VNM,False,2024-12-01,Other fossil,False,0.00,0.00,0.00,0.00,0.00,0.00,175.15,100.29,2024,11511.9
501,Viet Nam,VNM,False,2024-12-01,Solar,False,2.27,8.44,22.48,27.26,8.84,269.93,175.15,100.29,2024,11511.9
502,Viet Nam,VNM,False,2024-12-01,Total generation,True,26.91,100.00,266.47,308.26,100.00,3052.44,175.15,100.29,2024,11511.9


In [194]:
brentoil = pd.read_csv(r"data\raw\ins_data\brentoil_data.csv")
brentoil = normalize_to_month_start(brentoil, 'Date')
brentoil.rename(columns={'Date': 'date'}, inplace=True)

df_brent_clean = brentoil[['date', 'Price', 'Change']].copy()
df_brent_clean.columns = ['date', 'Oil_Price', 'Oil_Change_Percent']

if df_brent_clean['Oil_Change_Percent'].dtype == 'object':
    df_brent_clean['Oil_Change_Percent'] = df_brent_clean['Oil_Change_Percent'].str.replace('%', '').astype(float)
df_brent_clean.tail()

,date,Oil_Price,Oil_Change_Percent
79,2019-05-01,61.99,-13.97
80,2019-04-01,72.06,6.63
81,2019-03-01,67.58,1.92
82,2019-02-01,66.31,8.99
83,2019-01-01,60.84,12.62


In [195]:
emberVNcombine = pd.merge(emberVNcombine, df_brent_clean, on='date', how='left')
emberVNcombine.head()

,entity,entity_code,is_aggregate_entity,date,series,is_aggregate_series,generation_twh,generation_share_pct,generation_kwh_per_capita,generation_ytd_twh,generation_ytd_share_pct,generation_ytd_kwh_per_capita,IPI_Value,CPI_Value,Year,GDP_VND_Trillion,Oil_Price,Oil_Change_Percent
0,Viet Nam,VNM,False,2019-01-01,Coal,False,9.63,53.89,99.10,9.63,53.89,99.10,134.05,100.1,2019,7700.0,60.84,12.62
1,Viet Nam,VNM,False,2019-01-01,Gas,False,4.12,23.06,42.40,4.12,23.06,42.40,134.05,100.1,2019,7700.0,60.84,12.62
2,Viet Nam,VNM,False,2019-01-01,Hydro,False,3.39,18.97,34.89,3.39,18.97,34.89,134.05,100.1,2019,7700.0,60.84,12.62
3,Viet Nam,VNM,False,2019-01-01,Other fossil,False,0.00,0.00,0.00,0.00,0.00,0.00,134.05,100.1,2019,7700.0,60.84,12.62
4,Viet Nam,VNM,False,2019-01-01,Solar,False,0.00,0.00,0.00,0.00,0.00,0.00,134.05,100.1,2019,7700.0,60.84,12.62


In [196]:
df_fdi = pd.read_csv(r'data\raw\ins_data\fdi_data.csv', sep=';', decimal=',')
# 2. Xử lý cột Time (Từ "Dec-25" -> "2025-12-01")
df_fdi['Date'] = pd.to_datetime(df_fdi['Time'], format='%b-%y')
df_fdi = normalize_to_month_start(df_fdi, 'Date')

# 3. Làm sạch số liệu (Loại bỏ dấu phẩy ngăn cách hàng nghìn nếu có)
cols_to_fix = ['FDI giải ngân (lũy kế năm) (tỉ USD) (value - triệu USD)', 
               'FDI đăng kí (lũy kế năm) (tỉ USD) (value)']

for col in cols_to_fix:
    if df_fdi[col].dtype == 'object':
        df_fdi[col] = df_fdi[col].str.replace(',', '').astype(float)

# 4. GIẢI LŨY KẾ (Cực kỳ quan trọng)
# Sắp xếp theo thời gian trước khi tính
df_fdi = df_fdi.sort_values('Date')

# Tính giá trị thực tế theo tháng
# Group theo năm để tránh việc lấy tháng 1 năm nay trừ đi tháng 12 năm ngoái
df_fdi['FDI_Disbursed_Monthly(bilionUSD)'] = df_fdi.groupby(df_fdi['Date'].dt.year)[cols_to_fix[0]].diff().fillna(df_fdi[cols_to_fix[0]])
df_fdi['FDI_Registered_Monthly(bilionUSD)'] = df_fdi.groupby(df_fdi['Date'].dt.year)[cols_to_fix[1]].diff().fillna(df_fdi[cols_to_fix[1]])

# 5. Chỉ lấy các cột cần thiết để gộp
df_fdi_final = df_fdi[['Date', 'FDI_Disbursed_Monthly(bilionUSD)', 'FDI_Registered_Monthly(bilionUSD)']]
df_fdi_final.rename(columns={'Date': 'date'}, inplace=True)
df_fdi_final

┌──────────────────────────────── ⚠️ Warning ─────────────────────────────────┐
│ SettingWithCopyWarning in                                                   │
│ C:\Users\Admin\AppData\Local\Temp\ipykernel_10784\1782066438.py:25          │
│                                                                             │
│ A value is trying to be set on a copy of a slice from a DataFrame           │
│                                                                             │
│ See the caveats in the documentation:                                       │
│ https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#retur │
│ ning-a-view-versus-a-copy                                                   │
└─────────────────────────────────────────────────────────────────────────────┘


,date,FDI_Disbursed_Monthly(bilionUSD),FDI_Registered_Monthly(bilionUSD)
83,2019-01-01,1.55,1.907
82,2019-02-01,1.03,6.564
81,2019-03-01,1.54,2.334
80,2019-04-01,1.58,3.786
79,2019-05-01,1.60,2.146
...,...,...,...
4,2025-08-01,1.80,2.050
3,2025-09-01,3.40,2.400
2,2025-10-01,2.50,2.980
1,2025-11-01,2.30,2.170


In [197]:
emberVNcombine = pd.merge(emberVNcombine, df_fdi_final, on='date', how='left')
emberVNcombine


,entity,entity_code,is_aggregate_entity,date,series,is_aggregate_series,generation_twh,generation_share_pct,generation_kwh_per_capita,generation_ytd_twh,generation_ytd_share_pct,generation_ytd_kwh_per_capita,IPI_Value,CPI_Value,Year,GDP_VND_Trillion,Oil_Price,Oil_Change_Percent,FDI_Disbursed_Monthly(bilionUSD),FDI_Registered_Monthly(bilionUSD)
0,Viet Nam,VNM,False,2019-01-01,Coal,False,9.63,53.89,99.10,9.63,53.89,99.10,134.05,100.10,2019,7700.0,60.84,12.62,1.55,1.907
1,Viet Nam,VNM,False,2019-01-01,Gas,False,4.12,23.06,42.40,4.12,23.06,42.40,134.05,100.10,2019,7700.0,60.84,12.62,1.55,1.907
2,Viet Nam,VNM,False,2019-01-01,Hydro,False,3.39,18.97,34.89,3.39,18.97,34.89,134.05,100.10,2019,7700.0,60.84,12.62,1.55,1.907
3,Viet Nam,VNM,False,2019-01-01,Other fossil,False,0.00,0.00,0.00,0.00,0.00,0.00,134.05,100.10,2019,7700.0,60.84,12.62,1.55,1.907
4,Viet Nam,VNM,False,2019-01-01,Solar,False,0.00,0.00,0.00,0.00,0.00,0.00,134.05,100.10,2019,7700.0,60.84,12.62,1.55,1.907
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499,Viet Nam,VNM,False,2024-12-01,Hydro,False,11.29,41.95,111.80,98.13,31.83,971.70,175.15,100.29,2024,11511.9,74.64,3.90,3.67,6.850
500,Viet Nam,VNM,False,2024-12-01,Other fossil,False,0.00,0.00,0.00,0.00,0.00,0.00,175.15,100.29,2024,11511.9,74.64,3.90,3.67,6.850
501,Viet Nam,VNM,False,2024-12-01,Solar,False,2.27,8.44,22.48,27.26,8.84,269.93,175.15,100.29,2024,11511.9,74.64,3.90,3.67,6.850
502,Viet Nam,VNM,False,2024-12-01,Total generation,True,26.91,100.00,266.47,308.26,100.00,3052.44,175.15,100.29,2024,11511.9,74.64,3.90,3.67,6.850
